# Hidden pulls of the covariance systematics

A modeling knob that is not floated enters the fit through the systematic covariance. It is still
pulled by the data: a covariance term is a profiled linear parameter that is never reported. This
notebook computes those hidden pulls from existing fit outputs (no new fits) and uses them to
explain the nuisance-ladder result of notebook 25, where floating `NormCCMEC` on NuWro fake data
*raised* $\chi^2$ by about 12.

**Method.** PROfit's $\chi^2$ at the best fit is
$u^T (C + \Sigma)^{-1} u$ with $u = d - p$ (data minus the spline-only prediction), $C$ the statistical variance of the fit's statistic
(Neyman: ${\rm diag}(d)$; CNP: $3/(1/d+2/\mu)$, the default) and $\Sigma = \sum_i \Sigma_i$, $\Sigma_i = {\rm diag}(p)\,F_i\,{\rm diag}(p)$ the fractional
covariance of source $i$ scaled by the prediction. With $W = (C+\Sigma)^{-1}$:

* source $i$ shifts the prediction by $\delta_i = \Sigma_i W u$ and contributes $q_i = u^T W \Sigma_i W u$
  to $\chi^2_{\rm syst}$; the $q_i$ add up to $\chi^2_{\rm syst}$ exactly;
* for a rank-one source $F_i = c\,c^T$ (a linear knob) the hidden pull is $\hat x_i = (p\circ c)^T W u$
  in prior $\sigma$, and $q_i = \hat x_i^2$.

**Inputs.** $d$, the drawn best fit $b = p + \sum_i\delta_i$ and the per-bin covariance pulls come from
each ladder fit's `PROfile.root`; the per-source $F_i$ from the production `PROplot.root` of the same
family and fit (written with `--with-covar`), dropping the sources the ladder step floats.
$p$ is recovered from $b$ and the stored bin pulls. The $F_i$ are collapsed at the CV subchannel
mix, so the reconstruction is not exact; Section 1 checks it against PROfit's own split.

**Sign.** A covariance has no sign. The hidden pull is signed so that positive raises the total
predicted rate. For a normalization knob this is the knob's own sign; for other knobs it can be
the opposite.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import uproot
from IPython.display import display
from scipy.optimize import minimize_scalar

# Locate ma_zexp whether the notebook starts here or from axial_mass.
start = Path.cwd().resolve()
REPO = next(
    (parent / 'ma_zexp' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts/postfit_physical_parameters.py')
if str(REPO / 'python' / 'scripts') not in sys.path:
    sys.path.insert(0, str(REPO / 'python' / 'scripts'))

from postfit_physical_parameters import PUBLICATION_RC
from systematic_breakdown import SYST_GROUPS, xml_systematics
import make_nuisance_ladder_study_xmls as ladder

mpl.rcParams.update(PUBLICATION_RC)

## Controls

In [ ]:
OUTPUT_ROOT = Path(os.environ.get('AXIAL_MASS_OUTPUT_ROOT', '/nevis/hopper/data/epelaez/axial_mass_cnpmu_iterfrozen'))
STUDY_ROOT = Path(os.environ.get('NUISANCE_LADDER_STUDY_ROOT', OUTPUT_ROOT / 'nuisance_ladder_study_fit_results'))
FAMILIES = ('nuwro', 'opendata')               # Asimov closes at chi2 = 0, so every hidden pull is 0 there
FITS = list(ladder.FITS)                       # ma, minerva_k6_nuisance
VARIANTS = list(ladder.VARIANTS)               # n5 .. n10
LADDER_KNOBS = [name for name, _ in ladder.LADDER]   # knob floated at each step after n5
CHANNEL = 'nu_uBooNE_numuCC1p'
RANK_ONE = 0.99                                # leading eigenvalue fraction above which a source counts as a linear knob
# Statistic and covariance treatment of the fits being read. The statistical variance per bin is
#   neyman: d;  pearson: p;  CNP: 3 / (1/d + 2/mu), with mu the prediction WITHOUT spline shifts
# (PROfit's CNP takes mu from the physics-only CV; these fits have no physics parameters, so mu is the
# nominal CV whatever the best fit or the covariance reference is). With the iterated frozen
# covariance the systematic part is built at the reference, which is the best fit within TOL.
METRIC = os.environ.get('AXIAL_MASS_METRIC', 'CNP' if 'cnp' in str(OUTPUT_ROOT).lower() else 'neyman')
print(f'fits at {STUDY_ROOT}, statistic {METRIC}')
FIG_DIR = REPO / 'figs' / 'hidden_covariance_pulls'
TABLE_DIR = REPO / 'tables' / 'hidden_covariance_pulls'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

short = lambda name: name.removesuffix('_UBGenie')
TAG_GROUP = {tag: group for group, tags in SYST_GROUPS.items() for tag in tags}

## 1. Reconstruction and closure

`load_step` rebuilds $p$, $\Sigma$ and $W$ for one ladder fit. The floated parameters are read from
the fit's own MCMC chain, so the covariance contains exactly the sources that fit kept.

Closure: $\chi^2_{\rm stat}$ (residual of the drawn curve) and $\chi^2_{\rm syst}$ (sum of the squared
covariance-mode pulls) as PROfit stored them, against the same numbers from the reconstruction.

In [ ]:
def fractional_sources(family, fit):
    # Collapsed fractional covariance of every source in the production PROplot file.
    plot_file = OUTPUT_ROOT / f'{family}_fit_results' / fit / f'{fit}_v1_PROplot.root'
    with uproot.open(plot_file) as f:
        # Iterate rather than index: uproot parses ':' in 'collapsed_nu_uBooNE:0.02_cov' as a path.
        sources = {key.split(';')[0][len('collapsed_'):-len('_cov')]: hist.values().astype(float)
                   for key, hist in f['Covariance'].items(recursive=False)
                   if key.startswith('collapsed_') and key.split(';')[0].endswith('_cov')}
    if 'total_frac' not in sources:
        raise ValueError(f'{plot_file} has no per-source covariances; rerun its plot stage with --with-covar')
    sources.pop('total_frac')
    return sources


def stat_variance(data, prediction, cv):
    if METRIC.lower() == 'neyman':
        return data
    if METRIC.lower() == 'pearson':
        return prediction
    if METRIC.lower() == 'cnp':
        return 3.0 / (1.0 / data + 2.0 / cv)
    raise ValueError(f'unknown METRIC {METRIC}')


def read_chi2(log):
    import re
    found = re.findall(r'Global Best Fit chi\^2:\s*(\S+)', log.read_text(errors='replace')) if log.is_file() else []
    return float(found[-1]) if found else np.nan


def load_step(family, variant, fit):
    profile = STUDY_ROOT / family / variant / fit / f'{fit}_v1_PROfile.root'
    with uproot.open(profile) as f:
        floated = set(f[f'{fit}_v1_PROfile_mcmc_chain'].keys())
        data = f[f'{CHANNEL}_data2d'].values().reshape(-1).astype(float)
        drawn = f[f'{CHANNEL}_bestfit2d'].values().reshape(-1).astype(float)
        cv = f[f'{CHANNEL}_cv2d'].values().reshape(-1).astype(float)
        bin_pulls = f['covmode_bin_pulls'].values(axis='y')
        mode_pulls = f['covmode_pulls'].values(axis='y')
        parameters = dict(zip(f['global_fit_result'].axis().labels(), f['global_fit_result'].values()))
    sources = {name: F for name, F in SOURCES[family, fit].items() if name not in floated}
    F_total = sum(sources.values())
    # drawn = p + shift and bin_pull = shift / sqrt(Sigma_ii) with Sigma_ii = p_i^2 F_ii.
    prediction = drawn / (1.0 + bin_pulls * np.sqrt(np.diag(F_total)))
    variance = stat_variance(data, prediction, cv)
    C = np.diag(variance)
    W = np.linalg.inv(C + np.outer(prediction, prediction) * F_total)
    u = data - prediction
    return dict(family=family, variant=variant, fit=fit, data=data, drawn=drawn, prediction=prediction,
                sources=sources, F_total=F_total, C=C, W=W, u=u, parameters=parameters,
                stored_stat=float(np.sum((data - drawn) ** 2 / variance)), stored_syst=float(np.sum(mode_pulls ** 2)),
                pull_chi2=float(sum(value ** 2 for value in parameters.values())))


SOURCES = {(family, fit): fractional_sources(family, fit) for family in FAMILIES for fit in FITS}
STEPS = {(family, v, fit): load_step(family, v, fit) for family in FAMILIES for v in VARIANTS for fit in FITS}

rows = []
for key, s in STEPS.items():
    absolute = np.outer(s['prediction'], s['prediction']) * s['F_total']
    shift = absolute @ s['W'] @ s['u']
    rows.append(dict(family=key[0], fit=key[2], variant=key[1],
                     stat_stored=s['stored_stat'], stat_rebuilt=float(np.sum((s['u'] - shift) ** 2 / np.diag(s['C']))),
                     syst_stored=s['stored_syst'], syst_rebuilt=float(s['u'] @ s['W'] @ absolute @ s['W'] @ s['u']),
                     chi2_data=float(s['u'] @ s['W'] @ s['u']), chi2_pull=s['pull_chi2'],
                     chi2_log=read_chi2(STUDY_ROOT / key[0] / key[1] / key[2] / 'profile.log')))
CLOSURE = pd.DataFrame(rows).set_index(['family', 'fit', 'variant'])
CLOSURE['chi2_tot'] = CLOSURE.chi2_data + CLOSURE.chi2_pull
print(f"largest |rebuilt chi2 - PROfit's logged chi2|: {(CLOSURE.chi2_tot - CLOSURE.chi2_log).abs().max():.2f}")
worst = max((CLOSURE.stat_rebuilt / CLOSURE.stat_stored - 1).abs().max(), (CLOSURE.syst_rebuilt / CLOSURE.syst_stored - 1).abs().max())
print(f'largest relative difference between the stored and rebuilt chi2 terms: {100 * worst:.1f} %')
display(CLOSURE.style.format('{:.2f}'))

## 2. Hidden pulls of every covariance source

Per source: its share $q_i$ of $\chi^2_{\rm syst}$, the fraction of its variance in the leading
eigenmode (1 = a linear knob) and, for linear knobs, the signed hidden pull $\hat x_i$. The table
lists the sources with the largest share in any fit.

In [ ]:
def hidden_pulls(step):
    p, W, u = step['prediction'], step['W'], step['u']
    rows = []
    for name, F in step['sources'].items():
        eigenvalues, eigenvectors = np.linalg.eigh(F)
        trace = eigenvalues.sum()
        rank_one = eigenvalues[-1] / trace if trace > 0 else np.nan
        c = np.sqrt(max(eigenvalues[-1], 0.0)) * eigenvectors[:, -1]
        c = c * (np.sign(np.sum(p * c)) or 1.0)                  # positive pull raises the total rate
        share = float(u @ W @ (np.outer(p, p) * F) @ W @ u)
        rows.append(dict(source=short(name), chi2_share=share, rank_one=rank_one,
                         hidden_pull=float((p * c) @ W @ u) if rank_one > RANK_ONE else np.nan))
    return pd.DataFrame(rows)


PULLS = pd.concat([hidden_pulls(s).assign(family=key[0], variant=key[1], fit=key[2]) for key, s in STEPS.items()],
                  ignore_index=True)
PULLS.to_csv(TABLE_DIR / 'hidden_pulls.csv', index=False, float_format='%.4f')

view = PULLS[PULLS.variant.isin([VARIANTS[0], VARIANTS[-2]])]
top = view.groupby('source').chi2_share.max().sort_values(ascending=False).head(14).index
for quantity in ('chi2_share', 'hidden_pull'):
    print(quantity)
    display(view[view.source.isin(top)].pivot_table(index='source', columns=['family', 'fit', 'variant'], values=quantity)
            .reindex(top).style.format('{:.2f}', na_rep=''))

In [ ]:
# Hidden pulls of the modeling knobs in the production configuration (n5), NuWro against open data.
xsec_tags = set(SYST_GROUPS['Cross section'])
xml = xml_systematics(REPO / 'xml' / 'nuwro' / f'{FITS[0]}.xml')
xsec = {short(name) for name, spec in xml.items() if spec['tag'] in xsec_tags}
n5 = PULLS[(PULLS.variant == VARIANTS[0]) & (PULLS.fit == FITS[0]) & PULLS.source.isin(xsec) & PULLS.hidden_pull.notna()]
table = n5.pivot(index='source', columns='family', values='hidden_pull')
order = table.abs().max(axis=1).sort_values(ascending=False).head(15).index[::-1]
fig, ax = plt.subplots(figsize=(6.4, 5.2))
y = np.arange(len(order))
for offset, family, color in ((0.18, 'nuwro', '#eb6834'), (-0.18, 'opendata', '#2a78d6')):
    ax.barh(y + offset, table.loc[order, family], height=0.34, color=color, label={'nuwro': 'NuWro fake data', 'opendata': 'Open data'}[family])
ax.axvline(0, color='0.3', lw=0.8)
for x in (-1, 1):
    ax.axvline(x, color='0.6', lw=0.8, ls=':')
ax.set_yticks(y)
ax.set_yticklabels(order, fontsize=9)
ax.tick_params(axis='x', labelsize=9)
ax.set_xlabel(r'hidden pull [prior $\sigma$] (positive raises the rate)', fontsize=10)
ax.set_title(f'{FITS[0]}, {VARIANTS[0]}: covariance knobs', fontsize=9)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'hidden_pulls_n5.pdf')
plt.show()

## 3. What floating a knob should cost

Each ladder step moves one knob from the covariance to the fit. Starting from the fit *before* the
step, with every other parameter held at its best fit, the knob is given an explicit value $x$ and
$\chi^2(x) + x^2$ is minimized in two ways:

* **linear**: the prediction becomes $p(1 + x c)$ and the rest of the covariance stays
  ${\rm diag}(p)\,F'\,{\rm diag}(p)$. This is the textbook equivalence, so it returns the hidden pull
  and $\Delta\chi^2 = 0$ by construction;
* **rescaled**: as PROfit does when the knob is a spline, the remaining *fractional* covariance
  multiplies the shifted prediction, ${\rm diag}(p(1+xc))\,F'\,{\rm diag}(p(1+xc))$.

`observed` is the actual change of $\chi^2_{\rm tot}$ between the two fits. The real fit also
re-optimizes the other parameters, so it can only do better than the rescaled prediction.

In [ ]:
def explicit_knob(step, name):
    # chi2 as a function of the knob value for the linear and the rescaled treatment.
    p, d, C = step['prediction'], step['data'], step['C']
    F = step['sources'][name]
    eigenvalues, eigenvectors = np.linalg.eigh(F)
    c = np.sqrt(eigenvalues[-1]) * eigenvectors[:, -1]
    c = c * np.sign(np.sum(p * c))
    F_rest = step['F_total'] - F

    def chi2(prediction, scale):
        residual = d - prediction
        return float(residual @ np.linalg.solve(C + np.outer(scale, scale) * F_rest, residual))

    linear = lambda x: chi2(p * (1 + x * c), p) + x * x
    rescaled = lambda x: chi2(p * (1 + x * c), p * (1 + x * c)) + x * x
    return linear, rescaled, c


rows, CURVES = [], {}
for family in FAMILIES:
    for fit in FITS:
        for before, after, knob in zip(VARIANTS[:-1], VARIANTS[1:], LADDER_KNOBS):
            step = STEPS[family, before, fit]
            base = CLOSURE.loc[(family, fit, before)]
            linear, rescaled, c = explicit_knob(step, knob)
            best_linear = minimize_scalar(linear, bounds=(-4, 4), method='bounded')
            best_rescaled = minimize_scalar(rescaled, bounds=(-4, 4), method='bounded')
            CURVES[family, fit, knob] = (linear, rescaled, base.chi2_data)
            rows.append(dict(family=family, fit=fit, step=f'{before} -> {after}', knob=short(knob),
                             hidden_pull=best_linear.x, dchi2_linear=best_linear.fun - base.chi2_data,
                             pull_rescaled=best_rescaled.x, dchi2_rescaled=best_rescaled.fun - base.chi2_data,
                             observed=CLOSURE.loc[(family, fit, after), 'chi2_tot'] - base.chi2_tot,
                             largest_bin_response=float(np.abs(c).max())))
STEP_TABLE = pd.DataFrame(rows).set_index(['family', 'fit', 'step'])
STEP_TABLE.to_csv(TABLE_DIR / 'step_predictions.csv', float_format='%.4f')
display(STEP_TABLE.style.format({column: '{:.2f}' for column in STEP_TABLE.columns if column != 'knob'}))

In [ ]:
# chi2 against the knob value for the two steps that change chi2 on NuWro.
x = np.linspace(-3.5, 3.5, 141)
panels = [k for k in ('xsr_scc_Fv3_SCC', 'NormCCMEC_UBGenie') if ('nuwro', FITS[0], k) in CURVES]
fig, axes = plt.subplots(1, len(panels), figsize=(5.4 * len(panels), 4.0), squeeze=False)
for ax, knob in zip(axes[0], panels):
    linear, rescaled, base = CURVES['nuwro', FITS[0], knob]
    ax.plot(x, [linear(v) - base for v in x], color='#2a78d6', label='linear (covariance equivalent)')
    ax.plot(x, [rescaled(v) - base for v in x], color='#eb6834', label='rescaled (as a floated spline)')
    ax.axhline(0, color='0.5', lw=0.8, ls=':')
    row = STEP_TABLE.xs(('nuwro', FITS[0]), level=('family', 'fit'))
    row = row[row.knob == short(knob)].iloc[0]
    ax.axhline(row.observed, color='0.2', lw=0.9, ls='--', label='observed in the ladder')
    ax.set_xlabel(f'{short(knob)} [prior $\\sigma$] (positive raises the rate)', fontsize=10)
    ax.set_ylabel(r'$\Delta\chi^2$ vs. fit before the step', fontsize=10)
    ax.tick_params(labelsize=9)
    ax.set_ylim(-8, 30)
    ax.set_title(f'NuWro, {FITS[0]}', fontsize=9)
axes[0][0].legend(frameon=False, fontsize=7.5)
fig.tight_layout()
fig.savefig(FIG_DIR / 'floating_cost_nuwro.pdf')
plt.show()

## 4. Which sources lose their slack

For the `NormCCMEC` step on NuWro: the $\chi^2_{\rm syst}$ share of each covariance group before
the step, and the fractional change of the prediction that the hidden pull corresponds to in the
bins where the knob acts most.

In [ ]:
key = ('nuwro', VARIANTS[-2], FITS[0])
before = PULLS[(PULLS.family == key[0]) & (PULLS.variant == key[1]) & (PULLS.fit == key[2])].copy()
xml = xml_systematics(REPO / 'xml' / 'nuwro' / f'{FITS[0]}.xml')
group_of = {short(name): TAG_GROUP[spec['tag']] for name, spec in xml.items()}
before['group'] = before.source.map(group_of).fillna('other')
display(before.groupby('group').chi2_share.sum().sort_values(ascending=False).to_frame('chi2_syst share').style.format('{:.2f}'))

linear, rescaled, c = explicit_knob(STEPS[key], LADDER_KNOBS[-1])
row = STEP_TABLE.loc[(key[0], key[2], f'{VARIANTS[-2]} -> {VARIANTS[-1]}')]
change = row.hidden_pull * c
print(f"NormCCMEC hidden pull {row.hidden_pull:+.2f} sigma: prediction changes by {100 * change.min():.0f} % in the most "
      f"affected bin, by more than 20 % in {(np.abs(change) > 0.2).sum()} of {len(c)} bins; "
      f"the remaining fractional covariance in those bins shrinks by the same factor when the knob is floated.")

## 5. Hidden pull against floated pull

The same knob seen two ways along the nuisance ladder:

* **hidden**: the knob is still in the covariance (the fit *before* its ladder step); its pull is $\hat x = g^T W u$ with $g = p\circ c$, and its posterior width is $\sqrt{1 - g^T W g}$ (prior width 1);
* **floated**: the knob is a spline (the fit *after* its step); profile best fit and 68 % interval of the posterior chain.

Both are in the knob's own sign: the direction of the covariance eigenvector $c$ is fixed with the knob's $+1\sigma$ response from the `Splines` directory of the last ladder variant's `PROplot.root` (the cosine between the two is printed as a check). The floated value of a knob bounded by its spline grid or restricted to $[0, 1]$ cannot follow a hidden pull outside that range.

In [ ]:
def knob_response(family, fit, knob):
    """Collapsed +1 sigma fractional response of a knob, from the spline curves of the last variant."""
    with uproot.open(STUDY_ROOT / family / VARIANTS[-1] / fit / f'{fit}_v1_PROplot.root') as f:
        splines = f['Splines']
        n = sum(1 for key in splines.keys() if key.split(';')[0].startswith(f'{knob}_curve_'))
        response = []
        for i in range(n):
            x, y = splines[f'{knob}_curve_{i}'].values()
            response.append(np.interp(1.0, x, y) - np.interp(0.0, x, y))
    return np.asarray(response)


def floated(family, variant, fit, knob):
    with uproot.open(STUDY_ROOT / family / variant / fit / f'{fit}_v1_PROfile.root') as f:
        best = dict(zip(f['global_fit_result'].axis().labels(), f['global_fit_result'].values()))[short(knob)]
        q16, q84 = np.percentile(f[f'{fit}_v1_PROfile_mcmc_chain'][knob].array(library='np'), [16, 84])
    return best, q16, q84


rows = []
for family in FAMILIES:
    for fit in FITS:
        for before, after, knob in zip(VARIANTS[:-1], VARIANTS[1:], LADDER_KNOBS):
            step = STEPS[family, before, fit]
            p, W, u = step['prediction'], step['W'], step['u']
            eigenvalues, eigenvectors = np.linalg.eigh(step['sources'][knob])
            c = np.sqrt(eigenvalues[-1]) * eigenvectors[:, -1]
            response = knob_response(family, fit, knob)
            # The curves are per subchannel bin; every subchannel block is compared with c and the
            # best-aligned one fixes the sign.
            blocks = response.reshape(-1, len(c))
            cosines = [float(b @ c / (np.linalg.norm(b) * np.linalg.norm(c))) for b in blocks]
            cosine = max(cosines, key=abs)
            g = p * c * np.sign(cosine)
            best, q16, q84 = floated(family, after, fit, knob)
            rows.append(dict(family=family, fit=fit, knob=short(knob), hidden_in=before, floated_in=after,
                             hidden=float(g @ W @ u), hidden_sigma=float(np.sqrt(max(1 - g @ W @ g, 0.0))),
                             floated=best, floated_q16=q16, floated_q84=q84,
                             rank_one=eigenvalues[-1] / eigenvalues.sum(), cosine=abs(cosine)))
COMPARE = pd.DataFrame(rows)
COMPARE.to_csv(TABLE_DIR / 'hidden_vs_floated.csv', index=False, float_format='%.4f')
display(COMPARE.set_index(['family', 'fit', 'knob']).style.format({c: '{:.2f}' for c in COMPARE.columns if COMPARE[c].dtype.kind == 'f'}))

FAMILY_LABEL = {'nuwro': 'NuWro fake data', 'opendata': 'Open data', 'asimov': 'Asimov'}
FIT_LABEL = {'ma': r'$M_A$ fit', 'minerva_k6_nuisance': r'$z$ expansion, $\bar{\nu}$–H prior'}
KNOB_LABEL = {'VecFFCCQEshape': 'CCQE vector FF shape', 'FracPN_CCMEC': 'MEC pn fraction', 'xsr_scc_Fv3_SCC': r'second-class current $F_V^3$',
              'FrAbs_N': 'FrAbs N', 'NormCCMEC': 'MEC normalization'}
HIDDEN = dict(color='#0072B2', marker='o', offset=0.16, label='in the covariance (hidden pull)')
FLOATED = dict(color='#D55E00', marker='s', offset=-0.16, label='floated as a spline')
panels = [(family, fit) for fit in FITS for family in ('opendata', 'nuwro') if family in FAMILIES]
knobs = [short(k) for k in LADDER_KNOBS]
fig, axes = plt.subplots(len(FITS), 2, figsize=(7.0, 4.6), sharex=True, sharey=True, squeeze=False)
for ax, (family, fit) in zip(axes.ravel(), panels):
    part = COMPARE[(COMPARE.family == family) & (COMPARE.fit == fit)].set_index('knob')
    for i, knob in enumerate(knobs):
        y = len(knobs) - 1 - i
        row = part.loc[knob]
        ax.plot([row.hidden - row.hidden_sigma, row.hidden + row.hidden_sigma], [y + HIDDEN['offset']] * 2, color=HIDDEN['color'], lw=1.6, solid_capstyle='round')
        ax.plot(row.hidden, y + HIDDEN['offset'], HIDDEN['marker'], color=HIDDEN['color'], ms=5.5, mec='white', mew=0.8, label=HIDDEN['label'] if i == 0 else None)
        ax.plot([row.floated_q16, row.floated_q84], [y + FLOATED['offset']] * 2, color=FLOATED['color'], lw=1.6, solid_capstyle='round')
        ax.plot(row.floated, y + FLOATED['offset'], FLOATED['marker'], color=FLOATED['color'], ms=5.5, mec='white', mew=0.8, label=FLOATED['label'] if i == 0 else None)
    ax.axvline(0, color='0.75', lw=0.8, zorder=0)
    ax.set_yticks(range(len(knobs)))
    ax.set_yticklabels([KNOB_LABEL.get(k, k) for k in knobs[::-1]], fontsize=8)
    ax.set_ylim(-0.6, len(knobs) - 0.4)
    ax.set_title(f'{FAMILY_LABEL[family]}: {FIT_LABEL.get(fit, fit)}', fontsize=9)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='x', color='0.92', lw=0.6, zorder=0)
    ax.set_axisbelow(True)
fig.supxlabel(r'pull [prior $\sigma$]', fontsize=9)
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False, fontsize=8, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=(0, 0.01, 1, 0.945))
fig.savefig(FIG_DIR / 'hidden_vs_floated.pdf'); fig.savefig(FIG_DIR / 'hidden_vs_floated.png', dpi=200); plt.show()